# E_S2 — Deterministic Metrics Computation (multi-distribution, selected + other, heteroscedasticity)

Computes all deterministic metrics (no permutation) for each spread pattern, family group, and X distribution independently.

Reads from `F/output/E/E_S1/{spread}/{group}/{distribution}/` and saves to `F/output/E/E_S2/{spread}/{group}/{distribution}/`.

Metrics include: Pearson, Spearman, distance correlation/covariance,
MINE (MIC/MAS/MEV/MCN), LOWESS, GAM, bin-based, slope-based, distribution, etc.

In [7]:
from __future__ import annotations

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, ks_2samp, wasserstein_distance
from statsmodels.nonparametric.smoothers_lowess import lowess

try:
    from minepy import MINE
    HAS_MINEPY = True
except ImportError:
    HAS_MINEPY = False
    print('minepy not installed — MIC/MAS/MEV/MCN will be NaN.')

try:
    from pygam import LinearGAM, s as gam_s
    HAS_PYGAM = True
except ImportError:
    HAS_PYGAM = False
    print('pygam not installed — GAM metrics will be NaN.')

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings('ignore', category=np.RankWarning)

# ── Spread, group, and distribution settings ──
# SPREAD_PATTERNS = ['constant', 'increasing', 'decreasing', 'middle_high']

SPREAD_PATTERNS = [ 'increasing', 'decreasing', 'middle_high']
ENABLE_SPREAD_VARIATION = True
ACTIVE_SPREAD_PATTERNS = SPREAD_PATTERNS if ENABLE_SPREAD_VARIATION else ['constant']

# FAMILY_GROUPS = ['selected', 'other']
FAMILY_GROUPS = ['selected']
X_DISTRIBUTIONS = ['even', 'left_dense', 'right_dense', 'center_dense']
OVERWRITE = False

# ── Expensive metric switches (disabled by default) ──
ENABLE_DISTANCE = False
ENABLE_LOWESS = False
ENABLE_GAM = False

def locate_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in candidates:
        if (root / 'D' / 'D_S1_scatterplot_generation.ipynb').exists():
            return root
    raise FileNotFoundError('Could not locate repo root')

REPO_ROOT = locate_repo_root()
E_S1_OUTPUT_BASE = REPO_ROOT / 'F' / 'output' / 'E' / 'E_S1'
E_S2_OUTPUT_BASE = REPO_ROOT / 'F' / 'output' / 'E' / 'E_S2'

print(f'S1 input base: {E_S1_OUTPUT_BASE}')
print(f'S2 output base: {E_S2_OUTPUT_BASE}')
print(f'Spread patterns: {ACTIVE_SPREAD_PATTERNS}')
print(f'Family groups: {FAMILY_GROUPS}')
print(f'X distributions: {X_DISTRIBUTIONS}')
print(f'Expensive metrics: distance={ENABLE_DISTANCE}, '
      f'lowess={ENABLE_LOWESS}, gam={ENABLE_GAM}')

S1 input base: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/E/E_S1
S2 output base: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/E/E_S2
Spread patterns: ['constant', 'increasing', 'decreasing', 'middle_high']
Family groups: ['selected']
X distributions: ['even', 'left_dense', 'right_dense', 'center_dense']
Expensive metrics: distance=False, lowess=False, gam=False


## Metric Functions

In [8]:
def vectorised_pearson(x, y):
    xc = x - x.mean(axis=1, keepdims=True)
    yc = y - y.mean(axis=1, keepdims=True)
    num = (xc * yc).sum(axis=1)
    den = np.sqrt((xc**2).sum(axis=1) * (yc**2).sum(axis=1))
    return np.where(den > 0, num / den, np.nan)


def _rank_rows(arr):
    n_cases, n_points = arr.shape
    ranks = np.empty(arr.shape, dtype=np.float64)
    order = arr.argsort(axis=1)
    rows = np.arange(n_cases)[:, None]
    ranks[rows, order] = np.arange(1, n_points + 1, dtype=np.float64)
    return ranks


def vectorised_spearman(x, y):
    return vectorised_pearson(_rank_rows(x), _rank_rows(y))

In [9]:
def _to_valid(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    return x[valid], y[valid]

def _safe_div(a, b):
    if b == 0 or not np.isfinite(b):
        return np.nan
    return a / b

def _minmax01(v):
    v = np.asarray(v, dtype=float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return np.full_like(v, np.nan, dtype=float)
    return (v - lo) / (hi - lo)

def _endpoint_slope(x, y):
    if len(x) < 2: return np.nan
    return _safe_div(float(y[-1] - y[0]), float(x[-1] - x[0]))

def _polyfit_slope(x, y):
    if len(x) < 3 or np.std(x) == 0: return np.nan
    return float(np.polyfit(x, y, 1)[0])

def _segment_masks(n):
    i1, i2 = n // 3, 2 * n // 3
    early  = np.zeros(n, bool); early[:i1]   = True
    middle = np.zeros(n, bool); middle[i1:i2] = True
    late   = np.zeros(n, bool); late[i2:]    = True
    return early, middle, late

def _residual_sd(y, y_hat):
    r = y - y_hat
    return float(np.nanstd(r, ddof=1)) if len(r) >= 2 else np.nan

def _r2(y, y_hat):
    ss_res = float(np.nansum((y - y_hat)**2))
    ss_tot = float(np.nansum((y - np.nanmean(y))**2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

def _sign_changes(x_curve, y_curve, tol=1e-6):
    x_curve = np.asarray(x_curve, float)
    y_curve = np.asarray(y_curve, float)
    valid = np.isfinite(x_curve) & np.isfinite(y_curve)
    x_curve, y_curve = x_curve[valid], y_curve[valid]
    if len(x_curve) < 4: return np.nan
    order = np.argsort(x_curve)
    x_curve, y_curve = x_curve[order], y_curve[order]
    dx = np.diff(x_curve)
    dy = np.diff(y_curve)
    ok = dx != 0
    if ok.sum() < 3: return np.nan
    slopes = dy[ok] / dx[ok]
    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > tol] = 1
    signs[slopes < -tol] = -1
    nz = signs[signs != 0]
    return float(np.sum(nz[1:] != nz[:-1])) if len(nz) >= 2 else 0.0

def _make_bins(x, n_bins=10, bin_type='equal_width'):
    if bin_type == 'equal_width':
        return np.asarray(pd.cut(x, bins=n_bins, labels=False, include_lowest=True, duplicates='drop'), dtype=float)
    elif bin_type == 'equal_count':
        return np.asarray(pd.qcut(x, q=n_bins, labels=False, duplicates='drop'), dtype=float)
    raise ValueError(f'Unknown bin_type: {bin_type}')

In [10]:
def _double_center(a):
    a = a.reshape(-1, 1)
    dist = squareform(pdist(a))
    return dist - dist.mean(axis=0, keepdims=True) - dist.mean(axis=1, keepdims=True) + dist.mean()

def _distance_metrics(x, y):
    r = {'distance_covariance': np.nan, 'distance_correlation': np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return r
    A, B = _double_center(x), _double_center(y)
    dcov_xy = float(np.sqrt(max((A * B).mean(), 0)))
    dcov_xx = float(np.sqrt(max((A * A).mean(), 0)))
    dcov_yy = float(np.sqrt(max((B * B).mean(), 0)))
    r['distance_covariance'] = dcov_xy
    r['distance_correlation'] = _safe_div(dcov_xy, np.sqrt(dcov_xx * dcov_yy))
    return r

def _mine_metrics(x, y):
    empty = {'MIC': np.nan, 'MAS': np.nan, 'MEV': np.nan, 'MCN': np.nan, 'MIC_minus_r2': np.nan}
    if not HAS_MINEPY or len(x) < 5: return empty
    try:
        mine = MINE(alpha=0.6, c=15)
        mine.compute_score(x, y)
        mic = mine.mic()
        r = pearsonr(x, y)[0] if np.std(x) > 0 and np.std(y) > 0 else np.nan
        return {'MIC': mic, 'MAS': mine.mas(), 'MEV': mine.mev(), 'MCN': mine.mcn(),
                'MIC_minus_r2': mic - r**2 if np.isfinite(r) else np.nan}
    except Exception:
        return empty

def _slope_metrics(x, y, prefix='raw'):
    keys = []
    for method in ['endpoint', 'polyfit']:
        for seg in ['overall', 'early', 'middle', 'late']:
            keys.append(f'{prefix}_{method}_{seg}_slope')
    keys.append(f'{prefix}_segment_strength')
    empty = {k: np.nan for k in keys}
    if len(x) < 6: return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    em, mm, lm = _segment_masks(len(xs))
    segments = {'overall': (xs, ys), 'early': (xs[em], ys[em]),
                'middle': (xs[mm], ys[mm]), 'late': (xs[lm], ys[lm])}
    r = {}
    ep_seg_abs = []
    for seg_name, (sx, sy) in segments.items():
        ep = _endpoint_slope(sx, sy)
        pf = _polyfit_slope(sx, sy)
        r[f'{prefix}_endpoint_{seg_name}_slope'] = ep
        r[f'{prefix}_polyfit_{seg_name}_slope'] = pf
        if seg_name != 'overall':
            ep_seg_abs.append(abs(ep) if np.isfinite(ep) else np.nan)
    r[f'{prefix}_segment_strength'] = float(np.nanmean(ep_seg_abs))
    return r

def _standardized_slope_metrics(x, y):
    xn, yn = _minmax01(x), _minmax01(y)
    if np.any(np.isnan(xn)) or np.any(np.isnan(yn)):
        keys = []
        for method in ['endpoint', 'polyfit']:
            for seg in ['overall', 'early', 'middle', 'late']:
                keys.append(f'standardized_{method}_{seg}_slope')
        keys.append('standardized_segment_strength')
        return {k: np.nan for k in keys}
    return _slope_metrics(xn, yn, prefix='standardized')

def _bin_metrics(x, y, n_bins=10, bin_type='equal_width', min_count=5):
    prefix = f'{bin_type}_bin'
    r = {f'{prefix}_amplitude': np.nan, f'{prefix}_eta_squared': np.nan,
         f'{prefix}_buffer_width_mean': np.nan,
         f'{prefix}_early_buffer_width': np.nan, f'{prefix}_middle_buffer_width': np.nan,
         f'{prefix}_late_buffer_width': np.nan, f'{prefix}_n_valid_bins': 0}
    if len(x) < n_bins: return r
    try: bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception: return r
    df = pd.DataFrame({'x': x, 'y': y, 'bin': bins}).dropna()
    if df.empty: return r
    bin_stats = []
    for bid, g in df.groupby('bin', observed=True):
        if len(g) < min_count: continue
        yv = g['y'].values
        bw = float(np.nanpercentile(yv, 95) - np.nanpercentile(yv, 5))
        bin_stats.append({'bin': bid, 'x_mean': float(g['x'].mean()),
                          'y_mean': float(yv.mean()), 'buffer_width': bw, 'count': len(g)})
    if len(bin_stats) < 2: return r
    bdf = pd.DataFrame(bin_stats).sort_values('x_mean').reset_index(drop=True)
    r[f'{prefix}_amplitude'] = float(bdf['y_mean'].max() - bdf['y_mean'].min())
    y_global = float(df['y'].mean())
    ss_tot = float(np.sum((df['y'].values - y_global)**2))
    ss_bet = sum(row['count'] * (row['y_mean'] - y_global)**2 for _, row in bdf.iterrows())
    r[f'{prefix}_eta_squared'] = _safe_div(ss_bet, ss_tot)
    r[f'{prefix}_buffer_width_mean'] = float(np.nanmean(bdf['buffer_width']))
    r[f'{prefix}_n_valid_bins'] = len(bdf)
    nv = len(bdf)
    i1, i2 = nv // 3, 2 * nv // 3
    r[f'{prefix}_early_buffer_width'] = float(np.nanmean(bdf.iloc[:i1]['buffer_width']))
    r[f'{prefix}_middle_buffer_width'] = float(np.nanmean(bdf.iloc[i1:i2]['buffer_width']))
    r[f'{prefix}_late_buffer_width'] = float(np.nanmean(bdf.iloc[i2:]['buffer_width']))
    return r

def _x_coverage_metrics(x, n_bins=10):
    r = {'x_bin_count_cv': np.nan, 'x_uniform_ks_distance': np.nan}
    xf = x[np.isfinite(x)]
    if len(xf) < 3: return r
    lo, hi = xf.min(), xf.max()
    if hi > lo:
        xn = np.sort((xf - lo) / (hi - lo))
        n = len(xn)
        r['x_uniform_ks_distance'] = float(max(
            np.max(np.arange(1, n+1) / n - xn), np.max(xn - np.arange(0, n) / n)))
    if len(xf) >= n_bins:
        counts, _ = np.histogram(xf, bins=n_bins)
        mu = counts.mean()
        if mu > 0: r['x_bin_count_cv'] = float(np.std(counts, ddof=1) / mu)
    return r

def _lowess_metrics(x, y, frac=0.25):
    empty = {'lowess_residual_sd': np.nan, 'lowess_curve_amplitude': np.nan,
             'lowess_r2': np.nan, 'lowess_first_derivative_sign_changes': np.nan,
             'lowess_overall_slope': np.nan}
    if len(x) < 5: return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    fitted = lowess(ys, xs, frac=frac, return_sorted=True)
    x_fit, y_fit = fitted[:, 0], fitted[:, 1]
    y_pred = np.interp(xs, x_fit, y_fit)
    return {'lowess_residual_sd': _residual_sd(ys, y_pred),
            'lowess_curve_amplitude': float(np.nanmax(y_fit) - np.nanmin(y_fit)),
            'lowess_r2': _r2(ys, y_pred),
            'lowess_first_derivative_sign_changes': _sign_changes(x_fit, y_fit),
            'lowess_overall_slope': _endpoint_slope(x_fit, y_fit)}

def _gam_metrics(x, y, n_splines=10, lam=0.6, grid_size=200):
    empty = {'gam_residual_sd': np.nan, 'gam_curve_amplitude': np.nan,
             'gam_r2': np.nan, 'gam_first_derivative_sign_changes': np.nan,
             'gam_overall_slope': np.nan}
    if not HAS_PYGAM or len(x) < 10: return empty
    try:
        gam = LinearGAM(gam_s(0, n_splines=n_splines), lam=lam).fit(x.reshape(-1, 1), y)
        x_curve = np.linspace(x.min(), x.max(), grid_size)
        y_curve = gam.predict(x_curve.reshape(-1, 1))
        y_pred = np.interp(x, x_curve, y_curve)
        return {'gam_residual_sd': _residual_sd(y, y_pred),
                'gam_curve_amplitude': float(np.nanmax(y_curve) - np.nanmin(y_curve)),
                'gam_r2': _r2(y, y_pred),
                'gam_first_derivative_sign_changes': _sign_changes(x_curve, y_curve),
                'gam_overall_slope': _endpoint_slope(x_curve, y_curve)}
    except Exception:
        return empty

def _correlation_extra(x, y):
    r = {'covariance': np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return r
    r['covariance'] = float(np.cov(x, y, ddof=1)[0, 1])
    return r

def _distribution_metrics(x, y, n_bins=10, bin_type='equal_width'):
    prefix = f'{bin_type}_distribution'
    r = {f'{prefix}_ks_distance': np.nan, f'{prefix}_wasserstein_distance': np.nan}
    if len(x) < n_bins: return r
    try: bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception: return r
    df = pd.DataFrame({'x': x, 'y': y, 'bin': bins}).dropna()
    if df.empty: return r
    valid_bins = np.sort(df['bin'].unique())
    if len(valid_bins) < 3: return r
    nv = len(valid_bins)
    y_low = df[df['bin'].isin(valid_bins[:nv//3])]['y'].values
    y_high = df[df['bin'].isin(valid_bins[2*nv//3:])]['y'].values
    if len(y_low) < 2 or len(y_high) < 2: return r
    r[f'{prefix}_ks_distance'] = float(ks_2samp(y_low, y_high).statistic)
    r[f'{prefix}_wasserstein_distance'] = float(wasserstein_distance(y_low, y_high))
    return r

_Y_SCALE_METRICS = [
    'equal_width_distribution_wasserstein_distance', 'equal_count_distribution_wasserstein_distance',
    'equal_width_bin_amplitude', 'equal_width_bin_buffer_width_mean',
    'equal_width_bin_early_buffer_width', 'equal_width_bin_middle_buffer_width',
    'equal_width_bin_late_buffer_width',
    'equal_count_bin_amplitude', 'equal_count_bin_buffer_width_mean',
    'equal_count_bin_early_buffer_width', 'equal_count_bin_middle_buffer_width',
    'equal_count_bin_late_buffer_width',
    'lowess_residual_sd', 'lowess_curve_amplitude',
    'gam_residual_sd', 'gam_curve_amplitude',
]

def _ysd_normalized(y, metrics):
    y_sd = float(np.nanstd(y, ddof=1))
    r = {'y_sd': y_sd}
    for m in _Y_SCALE_METRICS:
        if m in metrics:
            r[f'{m}_div_y_sd'] = _safe_div(metrics[m], y_sd)
    return r

In [11]:
def compute_all_per_case(x, y):
    x, y = _to_valid(x, y)
    m = {}
    m.update(_correlation_extra(x, y))
    if ENABLE_DISTANCE:
        m.update(_distance_metrics(x, y))
    else:
        m.update({'distance_covariance': np.nan, 'distance_correlation': np.nan})
    m.update(_x_coverage_metrics(x))
    m.update(_distribution_metrics(x, y, bin_type='equal_width'))
    m.update(_distribution_metrics(x, y, bin_type='equal_count'))
    m.update(_slope_metrics(x, y, prefix='raw'))
    m.update(_standardized_slope_metrics(x, y))
    m.update(_mine_metrics(x, y))
    m.update(_bin_metrics(x, y, bin_type='equal_width'))
    m.update(_bin_metrics(x, y, bin_type='equal_count'))
    if ENABLE_LOWESS:
        m.update(_lowess_metrics(x, y))
    else:
        m.update({'lowess_residual_sd': np.nan, 'lowess_curve_amplitude': np.nan,
                  'lowess_r2': np.nan, 'lowess_first_derivative_sign_changes': np.nan,
                  'lowess_overall_slope': np.nan})
    if ENABLE_GAM:
        m.update(_gam_metrics(x, y))
    else:
        m.update({'gam_residual_sd': np.nan, 'gam_curve_amplitude': np.nan,
                  'gam_r2': np.nan, 'gam_first_derivative_sign_changes': np.nan,
                  'gam_overall_slope': np.nan})
    m.update(_ysd_normalized(y, m))
    m['n_valid'] = len(x)
    return m

print('Metric functions defined.')

Metric functions defined.


## Compute and save metrics (per spread × group × distribution)

In [12]:
for spread_pat in ACTIVE_SPREAD_PATTERNS:
    for group_name in FAMILY_GROUPS:
        for x_dist in X_DISTRIBUTIONS:
            s1_dir = E_S1_OUTPUT_BASE / spread_pat / group_name / x_dist
            s2_dir = E_S2_OUTPUT_BASE / spread_pat / group_name / x_dist
            out_path = s2_dir / 'metrics_full.parquet'
            tag = f'{spread_pat}/{group_name}/{x_dist}'

            if not (s1_dir / 'cases.csv').exists():
                print(f'[{tag}] S1 data not found, skipping.')
                continue
            if out_path.exists() and not OVERWRITE:
                print(f'[{tag}] Metrics already exist, skipping. '
                      'Set OVERWRITE=True to replace.')
                continue

            print(f'\n{"="*60}')
            print(f'Processing: {tag}')
            print(f'{"="*60}')

            cases_df = pd.read_csv(s1_dir / 'cases.csv', low_memory=False)
            pts = np.load(s1_dir / 'scatter_points.npz')
            x_all = pts['x']
            y_all = pts['y']
            n_cases = len(cases_df)
            print(f'Loaded: {n_cases:,} cases × {x_all.shape[1]} points')
            del pts

            t0 = time.time()
            x64 = x_all.astype(np.float64)
            y64 = y_all.astype(np.float64)
            pearson_all = vectorised_pearson(x64, y64)
            spearman_all = vectorised_spearman(x64, y64)
            del x64, y64
            print(f'Phase 1: Pearson + Spearman in {time.time()-t0:.1f}s')

            per_case_results = []
            t0 = time.time()
            for i in tqdm(range(n_cases), desc=f'{tag} per-case'):
                x = x_all[i].astype(np.float64)
                y = y_all[i].astype(np.float64)
                per_case_results.append(compute_all_per_case(x, y))
            elapsed = time.time() - t0
            print(f'Phase 2: {n_cases:,} cases in {elapsed/60:.1f} min')

            metrics_df = pd.DataFrame(per_case_results)
            metrics_df.insert(0, 'case_id', cases_df['case_id'].values)
            metrics_df['pearson_r'] = pearson_all
            metrics_df['spearman_rho'] = spearman_all

            s2_dir.mkdir(parents=True, exist_ok=True)
            metrics_df.to_parquet(out_path, index=False)
            print(f'Saved {out_path.relative_to(E_S2_OUTPUT_BASE)}  '
                  f'({len(metrics_df):,} rows × {len(metrics_df.columns)} cols)')

            del cases_df, x_all, y_all, pearson_all, spearman_all
            del per_case_results, metrics_df

print('\n=== Summary ===')
for spread_pat in ACTIVE_SPREAD_PATTERNS:
    print(f'\n=== {spread_pat} ===')
    for group_name in FAMILY_GROUPS:
        print(f'  --- {group_name} ---')
        for dist in X_DISTRIBUTIONS:
            p = E_S2_OUTPUT_BASE / spread_pat / group_name / dist / 'metrics_full.parquet'
            if p.exists():
                n = len(pd.read_parquet(p, columns=['case_id']))
                print(f'    {dist:15s}: {n:,} rows, '
                      f'{p.stat().st_size / 1024**2:.1f} MiB')
            else:
                print(f'    {dist:15s}: not computed')

[constant/selected/even] Metrics already exist, skipping. Set OVERWRITE=True to replace.
[constant/selected/left_dense] Metrics already exist, skipping. Set OVERWRITE=True to replace.
[constant/selected/right_dense] Metrics already exist, skipping. Set OVERWRITE=True to replace.
[constant/selected/center_dense] Metrics already exist, skipping. Set OVERWRITE=True to replace.

Processing: increasing/selected/even
Loaded: 95,440 cases × 500 points
Phase 1: Pearson + Spearman in 3.4s


increasing/selected/even per-case:   1%|          | 486/95440 [00:08<26:32, 59.64it/s]


KeyboardInterrupt: 